In [6]:
#survivors on Titanic
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier

train_data = pd.read_csv("/home/vophilong/Documents/2526_2A_DeepLearning/lab_part_2/lab3/lab3/titanic/train.csv")
test_data = pd.read_csv("/home/vophilong/Documents/2526_2A_DeepLearning/lab_part_2/lab3/lab3/titanic/test.csv")

In [7]:
print(train_data.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [8]:
print(train_data.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


#### Chuẩn bị data

In [9]:
# Tạo nhãn y_train từ cột "Survived" và loại bỏ cột này khỏi dữ liệu huấn luyện
y_train = train_data["Survived"]
train_data.drop(labels="Survived", axis=1, inplace=True)

# chuẩn bị data bằng cách gộp dữ liệu huấn luyện và kiểm tra lại với nhau để xử lý các cột giống nhau
full_data = pd.concat([train_data, test_data], axis=0)

# Xóa các cột không cần thiết và xử lý dữ liệu
drop_columns = ["Name", "Age", "SibSp", "Ticket", "Cabin", "Parch", "Embarked"]
full_data.drop(labels=drop_columns, axis=1, inplace=True)

full_data = pd.get_dummies(full_data, columns=["Sex"])
full_data.fillna(value=0.0, inplace=True)

X_train = full_data.values[0:891]
X_test = full_data.values[891:]

# Chuẩn hóa dữ liệu bằng MinMaxScaler để đưa tất cả các giá trị về khoảng [0, 1]
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Chia dữ liệu huấn luyện thành tập huấn luyện và tập kiểm tra (validation set) với tỷ lệ 70% huấn luyện và 30% kiểm tra

state = 12  
test_size = 0.30  
  
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=test_size, random_state=state)

### Xây dựng bộ phân loại Gradient Boosting – tối ưu hóa learning rate (tốc độ học)

In [10]:
lr_list = [0.05, 0.075, 0.1, 0.25, 0.5, 0.75, 1]
# Huấn luyện mô hình Gradient Boosting với các giá trị learning rate khác nhau và đánh giá hiệu suất trên tập huấn luyện và tập kiểm tra
for learning_rate in lr_list:
    gb_clf = GradientBoostingClassifier(n_estimators=20, learning_rate=learning_rate, max_features=2, max_depth=2, random_state=0)
    gb_clf.fit(X_train, y_train)

    print("Learning rate: ", learning_rate)
    print("Accuracy score (training): {0:.3f}".format(gb_clf.score(X_train, y_train)))
    print("Accuracy score (validation): {0:.3f}".format(gb_clf.score(X_val, y_val)))
    


Learning rate:  0.05
Accuracy score (training): 0.801
Accuracy score (validation): 0.731
Learning rate:  0.075
Accuracy score (training): 0.814
Accuracy score (validation): 0.731
Learning rate:  0.1
Accuracy score (training): 0.812
Accuracy score (validation): 0.724
Learning rate:  0.25
Accuracy score (training): 0.835
Accuracy score (validation): 0.750
Learning rate:  0.5
Accuracy score (training): 0.864
Accuracy score (validation): 0.772
Learning rate:  0.75
Accuracy score (training): 0.875
Accuracy score (validation): 0.754
Learning rate:  1
Accuracy score (training): 0.875
Accuracy score (validation): 0.739


### Tạo (sinh) các dự đoán.

In [11]:
# Chọn một giá trị learning rate cụ thể (ví dụ: 0.5) để huấn luyện mô hình và đánh giá hiệu suất trên tập kiểm tra
gb_clf2 = GradientBoostingClassifier(n_estimators=20, learning_rate=0.5, max_features=2, max_depth=2, random_state=0)
gb_clf2.fit(X_train, y_train)
predictions = gb_clf2.predict(X_val)

print("Confusion Matrix:")
print(confusion_matrix(y_val, predictions))

print("Classification Report")
print(classification_report(y_val, predictions))

Confusion Matrix:
[[142  19]
 [ 42  65]]
Classification Report
              precision    recall  f1-score   support

           0       0.77      0.88      0.82       161
           1       0.77      0.61      0.68       107

    accuracy                           0.77       268
   macro avg       0.77      0.74      0.75       268
weighted avg       0.77      0.77      0.77       268



### Ví dụ minh họa đơn giản 1b (XGBoost)

In [14]:
# Huấn luyện mô hình XGBoost và đánh giá hiệu suất trên tập kiểm tra
from xgboost import XGBClassifier

xgb_clf = XGBClassifier()
xgb_clf.fit(X_train, y_train)

score = xgb_clf.score(X_val, y_val)
print(score)

0.7313432835820896


### Ví dụ minh họa đơn giản 2a

In [17]:
# Đánh giá mô hình XGBoost trên tập kiểm tra
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
# load data
dataset = loadtxt('/home/vophilong/Documents/2526_2A_DeepLearning/lab_part_2/lab3/lab3/diabetes.csv', delimiter=",")
# split data into X and y
X = dataset[:,0:8]
Y = dataset[:,8]
# CV model
model = XGBClassifier()
kfold = KFold(n_splits=10, shuffle=True, random_state=7)
results = cross_val_score(model, X, Y, cv=kfold)
print("Accuracy: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))

Accuracy: 73.97% (5.53%)


### Ví dụ minh họa đơn giản 2b

In [20]:
# Đánh giá mô hình XGBoost trên tập kiểm tra với StratifiedKFold
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
# load data
dataset = loadtxt('/home/vophilong/Documents/2526_2A_DeepLearning/lab_part_2/lab3/lab3/diabetes.csv', delimiter=",")
# split data into X and y
X = dataset[:,0:8]
Y = dataset[:,8]
# CV model
model = XGBClassifier()

kfold = KFold(n_splits=10, shuffle=True, random_state=7)
results = cross_val_score(model, X, Y, cv=kfold)
print("Accuracy: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))

Accuracy: 73.97% (5.53%)
